In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

2026-02-13 14:53:30.679034: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-13 14:53:30.682389: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [3]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="16G",#memory per slurm job
        processes=1,#dask workers per slurm job,
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=6:00:00",
            f"--output=worker_%j.out"]
    )
    #cluster.scale(jobs=20)
    cluster.scale(jobs=20)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2026-02-13 14:53:42,487 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2026-02-13 14:53:42,488 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2026-02-13 14:53:42,489 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2026-02-13 14:53:42,490 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB


In [4]:
client.dashboard_link

'http://127.0.0.1:8787/status'

In [5]:
primordial=scm.ortho.load(client,data_root/"seelig","ortho_seelig_v2")

In [6]:
primordial.compute_model_qc()

In [7]:
import pandas as pd
import numpy as np

vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
gt_cell_type=pd.concat(vals)

In [8]:
#casting away from sparse, since it's not sparse anymore
gt_cell_type["mu"] = gt_cell_type["mu"].astype(float)

#this doesn't even work
gt_cell_type["cre_id"]=gt_cell_type["cre_id"].astype(str)
gt_cell_type["cell_type"]=gt_cell_type["cell_type"].astype(str)

In [9]:
gt_cell_type

,cre_id,mu,cell_type
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,0.006619,reference
1,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,0.794015,reference
2,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,0.007987,reference
3,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,0.032403,reference
4,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,0.035598,reference
...,...,...,...
1272,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,0.012286,K562
1273,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,0.005746,K562
1274,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,0.005747,K562
1275,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,0.003171,K562


In [16]:
combo_counts=gt_cell_type[gt_cell_type["cre_id"]!="reference"].drop(columns=["mu"]).groupby("cell_type").nunique()
max_tfection=max(combo_counts["cre_id"])
combo_counts

,cre_id
cell_type,
K562,1276
reference,1329


In [10]:
gt_cell_type=gt_cell_type[gt_cell_type["cre_id"]!="reference"]

In [11]:
real_means=gt_cell_type.drop(columns=["cre_id","cell_type"])
real_means

,mu
0,0.006619
1,0.794015
2,0.007987
3,0.032403
4,0.035598
...,...
1271,0.002576
1272,0.012286
1273,0.005746
1274,0.005747


In [12]:
num_cell_types=len(gt_cell_type["cell_type"].unique())

In [13]:
num_cell_types

2

In [14]:
synth_cre_names=[f"synthcre_{i}" for i in range(0,len(real_means))]

parts=[]
for i in range(0,num_cell_types):
    working=real_means.sample(frac=1)
    working["cre_id"]=synth_cre_names
    working["cell_type"]=f"ct_{i}"
    parts.append(working)

cartesian=pd.concat(parts).sample(frac=1).reset_index(drop=True)
cartesian

,mu,cre_id,cell_type
0,0.005478,synthcre_209,ct_1
1,0.014605,synthcre_147,ct_0
2,0.014606,synthcre_1640,ct_0
3,0.035876,synthcre_2483,ct_1
4,0.010954,synthcre_2299,ct_1
...,...,...,...
5205,0.003171,synthcre_497,ct_0
5206,0.003765,synthcre_1378,ct_1
5207,0.022133,synthcre_272,ct_0
5208,0.019025,synthcre_738,ct_0


In [15]:
#make 70% inactive
minP=scm.SHENDURE_BOUNDS.reference_activity
num_inactive=int(len(cartesian)*0.7)
cartesian.loc[cartesian.index[:num_inactive],"mu"]=minP
cartesian


,mu,cre_id,cell_type
0,0.019311,synthcre_209,ct_1
1,0.019311,synthcre_147,ct_0
2,0.019311,synthcre_1640,ct_0
3,0.019311,synthcre_2483,ct_1
4,0.019311,synthcre_2299,ct_1
...,...,...,...
5205,0.003171,synthcre_497,ct_0
5206,0.003765,synthcre_1378,ct_1
5207,0.022133,synthcre_272,ct_0
5208,0.019025,synthcre_738,ct_0


In [17]:
#now we pick max_tfection CREs to proceed with, because this is the number of unique cre_id 
#found in the the cell type with the most unique CRE ids (reference, or PSC, in shend)

chosen_cre_id=np.random.choice(cartesian["cre_id"].unique(),max_tfection,replace=False)
sampled_nbs=cartesian[cartesian["cre_id"].isin(chosen_cre_id)]
sampled_nbs


,mu,cre_id,cell_type
0,0.019311,synthcre_209,ct_1
1,0.019311,synthcre_147,ct_0
2,0.019311,synthcre_1640,ct_0
3,0.019311,synthcre_2483,ct_1
8,0.019311,synthcre_1211,ct_1
...,...,...,...
5204,0.008719,synthcre_1786,ct_0
5205,0.003171,synthcre_497,ct_0
5206,0.003765,synthcre_1378,ct_1
5207,0.022133,synthcre_272,ct_0


In [18]:
client.close()
cluster.close()

2026-02-13 15:15:03,440 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker.py", line 1273, in heartbeat
    response = await retry_operation(
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/utils_comm.py", line 395, in retry
    return await coro()
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-pa